In [1]:
#%%
# 產業類別查詢 
#%%

import requests
from bs4 import BeautifulSoup
import re
from fake_useragent import UserAgent
import io
import os
import json
import pandas as pd
import numpy as np
import datetime
import time, random
from datetime import datetime
import random
import sqlite3
import unicodedata
import json
import yfinance as yf
import tempfile

import plotly.figure_factory as ff
import plotly.graph_objects as go
import plotly.express as px


pd.options.display.float_format = '{:.2f}'.format

UA_LIST = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:119.0) Gecko/20100101 Firefox/119.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7; rv:117.0) Gecko/20100101 Firefox/117.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36 Edg/119.0.0.0",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 16_0 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.0 Mobile/15E148 Safari/604.1",
    "Mozilla/5.0 (Linux; Android 12; Pixel 6 Pro) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.5845.61 Mobile Safari/537.36",
]


cookies_pool = [
    "_ga=GA1.1.291984994.1757777879; _ga_EJ69TXS89Q=GS2.1.s1757777879$o1$g0$t1757777879$j60$l0$h0",
    "_ga=GA1.1.647035994.1757777961; _ga_EJ69TXS89Q=GS2.1.s1757777960$o1$g1$t1757777982$j38$l0$h0",
    "_ga=GA1.1.2086850365.1757778059; _ga_EJ69TXS89Q=GS2.1.s1757778059$o1$g0$t1757778059$j60$l0$h0"
]

# Streamlit 讀取相關資料

In [7]:
#%%
# 同性質產業
# 下載github db 再讀取

def download_and_save_industry_db(url):
    response = requests.get(url)

    # 將數據保存到臨時文件中
    temp_file = tempfile.NamedTemporaryFile(delete=False)
    temp_file.write(response.content)
    temp_file_path = temp_file.name
    
    return temp_file_path


def download_sqlite_from_github(industry_db_path):
        response = requests.get(industry_db_path)
        
        if response.status_code != 200:
            raise Exception(f"下載失敗，網址：{industry_db_path}")
        
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".sqlite3")
        temp_file.write(response.content)
        temp_file.close()
        return temp_file.name


def get_stock_code_industry(stock_id, url):

    # 下載並保存數據庫文件
    temp_file_path = download_and_save_industry_db(url)

    conn = sqlite3.connect(temp_file_path)
    cursor = conn.cursor()

    # 提取公司名稱
    cursor.execute(f"SELECT 公司名稱 FROM industry WHERE 公司代號={stock_id};")
    result = cursor.fetchone()
    if result:
        stock_name = result[0]
        
    # 提取上市櫃
    cursor.execute(f"SELECT 上市櫃 FROM industry WHERE 公司代號={stock_id};")
    result2 = cursor.fetchone()
    if result2:
        cm_otc = result2[0]
    
    # 提取公司產業別
    cursor.execute(f"SELECT 產業類別提取 FROM industry WHERE 公司代號={stock_id};")
    result3 = cursor.fetchone()
    if result3:
        stock_industry = result3[0]

        # 使用提取的產業別，get相關產業所有資料
        cursor.execute(f"SELECT 公司代號, 公司名稱, 上市櫃 FROM industry WHERE 產業類別提取='{stock_industry}';")
        related_data = cursor.fetchall()

    cursor.close()
    conn.close()
    
    
    return stock_id, stock_name, cm_otc, stock_industry, related_data



In [5]:
stock_id = '1264'
url = f'https://github.com/06Cata/Taiwan_Stock/raw/main/industry.sqlite3'
stock_id, stock_name, cm_otc, stock_industry, related_data = get_stock_code_industry(stock_id, url)

print(stock_id)
print(stock_name)
print(cm_otc)
print(stock_industry)
print(related_data)

('德麥',)
1264
德麥
上櫃
食品工業
[('1201', '味全', '上市'), ('1203', '味王', '上市'), ('1210', '大成', '上市'), ('1213', '大飲', '上市'), ('1215', '卜蜂', '上市'), ('1216', '統一', '上市'), ('1217', '愛之味', '上市'), ('1218', '泰山', '上市'), ('1219', '福壽', '上市'), ('1220', '台榮', '上市'), ('1225', '福懋油', '上市'), ('1227', '佳格', '上市'), ('1229', '聯華', '上市'), ('1231', '聯華食', '上市'), ('1232', '大統益', '上市'), ('1233', '天仁', '上市'), ('1234', '黑松', '上市'), ('1235', '興泰', '上市'), ('1236', '宏亞', '上市'), ('1702', '南僑', '上市'), ('1737', '臺鹽', '上市'), ('3054', '立萬利', '上市'), ('7780', '大研生醫', '上市'), ('1264', '德麥', '上櫃'), ('1796', '金穎生技', '上櫃'), ('4205', '中華食', '上櫃'), ('4207', '環泰', '上櫃'), ('1294', '漢田生技', '上櫃'), ('6846', '綠茵', '上櫃'), ('1295', '生合', '上櫃'), ('7743', '金利食安', '上櫃')]


In [9]:
industry_name = 'industry'
industry_db_path = f'https://github.com/06Cata/Taiwan_Stock/raw/main/industry.sqlite3'
industry_db_path_download = download_sqlite_from_github(industry_db_path)

with sqlite3.connect(industry_db_path_download) as conn_indusrty:
        industry_num_sorted = pd.read_sql(f"SELECT * FROM {industry_name}", conn_indusrty)
        print(industry_num_sorted.head(1))
        
conn_indusrty.close()

industry_num_sorted

  產業類別提取  公司代號 公司名稱      產業類別 上市櫃
0   水泥工業  1101   台泥  產業別：水泥工業  上市


,產業類別提取,公司代號,公司名稱,產業類別,上市櫃
0,水泥工業,1101,台泥,產業別：水泥工業,上市
1,水泥工業,1102,亞泥,產業別：水泥工業,上市
2,水泥工業,1103,嘉泥,產業別：水泥工業,上市
3,水泥工業,1104,環泥,產業別：水泥工業,上市
4,水泥工業,1108,幸福,產業別：水泥工業,上市
...,...,...,...,...,...
1786,居家生活,6728,上洋,產業別：居家生活,上櫃
1787,居家生活,8066,來思達,產業別：居家生活,上櫃
1788,居家生活,8433,弘帆,產業別：居家生活,上櫃
1789,居家生活,8941,關中,產業別：居家生活,上櫃


In [ ]:
def diff_industry_type()